[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/087.%20The%20Hub-and-Spoke%20Network%20Design%20Problem/P87-Tier-8_executed.ipynb)

---


# 87. The Hub-and-Spoke Network Design Problem

## Tier 8: A Value-Aligned and Ethical Framework

This notebook develops a small, illustrative hub-and-spoke network model that scores candidate networks using stakeholder utilities and ethical constraint penalties. It follows a candidate-generation, evaluation, and best-candidate-selection sequence. The candidate generator is stochastic, so the selected solution is the best among the candidates generated in a run; it is not a proof of a global optimum.

### Purpose, inputs, and outputs

- **Purpose:** Compare candidate hub configurations while accounting for economic, service, environmental, and social indicators.
- **Inputs:** Synthetic coordinates, demand, hub costs, capacities, emissions factors, and stakeholder-related data generated in the code.
- **Outputs:** A selected candidate, its stakeholder utilities, constraint evaluations, a trade-off summary, and visualizations.
- **Solution sequence:** Generate candidates, assign each non-hub node to a hub, evaluate each candidate, select the highest-scoring candidate, and inspect its impacts.

### Scope and modeling assumptions

The network contains 20 illustrative locations. Coordinates are used to calculate great-circle distances; they do not describe a validated operational network. Demand and other inputs are synthetic, and random values are not seeded, so repeated runs can produce different data and candidates. Transport is represented by a distance-based proxy with fixed leg multipliers. Several social and operational indicators are simplified proxies, not measured outcomes. The model does not establish legal compliance, prove ethical acceptability, or estimate real-world carbon reductions.

Constraint types label the intended importance of a policy, but this implementation uses penalties for every violation rather than excluding infeasible candidates. Thus, a constraint labelled `hard` is not a mathematical feasibility guarantee. The compliance percentage counts constraints that meet their thresholds; it is distinct from the penalty-adjusted objective score.

### Stakeholders and principles

The model represents shareholders, employees, customers, the environment, and the community. Each stakeholder has a weighted utility assembled from network performance indicators. Sustainability, equity, transparency, and responsibility are represented as named principles. In this example, the five operational constraints are evaluated directly; the named principles provide context and are not additional optimization constraints.

### Candidate evaluation model

For candidate network \(s\), stakeholder \(g\)'s utility is the weighted sum of the performance indicators associated with that stakeholder:

$$
U_g(s) = \sum_{m \in M_g} w_{gm} P_m(s),
$$

where \(M_g\) is the set of indicators used for stakeholder \(g\), \(w_{gm}\) is the indicator weight, and \(P_m(s)\) is its normalized performance value. The overall stakeholder utility is the priority-weighted sum of stakeholder utilities.

For a constraint with measured value \(v_k(s)\) and threshold \(t_k\), the relative shortfall is zero when the requirement is met and otherwise is the shortfall divided by the threshold. For an upper-bound constraint, the relative violation is instead the excess divided by the threshold. The penalty multiplies this relative violation by the constraint's penalty factor and by a type-specific multiplier. The candidate score is stakeholder utility minus the total penalty:

$$
\begin{aligned}
R_k(s) &= \begin{cases}
\max\!\left(0, \frac{t_k-v_k(s)}{t_k}\right), & \text{for a lower bound},\\
\max\!\left(0, \frac{v_k(s)-t_k}{t_k}\right), & \text{for an upper bound},
\end{cases}\\
\Pi_k(s) &= R_k(s)\,q_k\,\lambda_k,\\
S(s) &= \sum_g p_g U_g(s) - \sum_k \Pi_k(s).
\end{aligned}
$$

Here, \(q_k\) is the constraint penalty factor, \(\lambda_k\) is 1 for a hard-labelled constraint, 0.5 for a soft constraint, and 0.25 for a preference, and \(p_g\) is stakeholder \(g\)'s priority. Relative violations make penalties dimensionless even when the constraint values use different units. The implementation is in `_evaluate_ethical_compliance` and `_calculate_overall_score`.

### Hub assignment and candidate search

A candidate specifies a set of hubs. Every hub is assigned to itself; each other location is assigned to the hub with the smallest distance-based assignment cost. A 10% assignment-cost discount is applied when a node and a candidate hub are in the same modeled region. This is a modeling preference, not a reduction in physical distance.

The search generates five candidates for each hub count from three through seven. Hub locations are sampled from a pool of high-scoring locations, where the score reflects employment impact, community need, wage similarity, and average route emissions. The candidate set is therefore stochastic and restricted; selecting its highest-scoring member does not establish a global optimum.

For \(n\) nodes and \(C\) candidates, the pairwise flow calculations take \(O(Cn^2)\) time. Candidate generation and evaluation are finite, but this is not exhaustive enumeration of all hub subsets. The central code functions are `_generate_candidate_solutions`, `_calculate_spoke_assignments`, `_calculate_network_performance`, and `optimize_value_aligned_network`.

### Reading the results

A higher score means a better result under the specified synthetic indicators, weights, thresholds, and penalties; it is not a universal measure of ethical quality. Inspect each constraint separately: a high overall score does not imply that every threshold is met. The emissions calculation is reported in metric tonnes after converting from kilograms. The carbon performance indicator is a normalized proxy, not a measured percentage reduction against a real baseline.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class StakeholderGroup:
    """A stakeholder group, its utility weights, and its priority."""

    name: str
    interests: List[str]
    utility_weights: Dict[str, float]
    constraints: List[str]
    priority: float


@dataclass
class EthicalPrinciple:
    """A named principle used to describe the framework's values."""

    name: str
    description: str
    constraints: List[str]
    metrics: List[str]
    minimum_threshold: float
    penalty_weight: float


@dataclass
class EthicalConstraint:
    """A threshold policy evaluated with a direction-aware penalty."""

    name: str
    type: str
    description: str
    evaluation_function: str
    threshold: float
    penalty_factor: float
    sense: str = "lower"


@dataclass
class ValueAlignedSolution:
    """The selected network and its evaluation summaries."""

    hub_configuration: List[int]
    spoke_assignments: Dict[int, int]
    stakeholder_utilities: Dict[str, float]
    ethical_compliance: Dict[str, object]
    overall_score: float
    trade_offs: Dict[str, float]
    explanation: str


class ValueAlignedEthicalFramework:
    """Score sampled hub-and-spoke candidates using value-related indicators."""

    def __init__(self, num_nodes=20):
        self.num_nodes = num_nodes
        self.stakeholders = self._initialize_stakeholders()
        self.ethical_principles = self._initialize_ethical_principles()
        self.ethical_constraints = self._initialize_ethical_constraints()
        self._initialize_network_data()
        self.solution_history = []
        self.compliance_history = []

    def _initialize_stakeholders(self):
        return {
            "shareholders": StakeholderGroup(
                name="Shareholders",
                interests=["profitability", "market_share", "dividend_growth", "stock_performance"],
                utility_weights={
                    "cost_efficiency": 0.4,
                    "revenue_growth": 0.3,
                    "market_expansion": 0.2,
                    "risk_management": 0.1,
                },
                constraints=["minimum_roi", "debt_ratio_limit", "compliance_requirements"],
                priority=0.25,
            ),
            "employees": StakeholderGroup(
                name="Employees",
                interests=["job_security", "fair_wages", "working_conditions", "career_development"],
                utility_weights={
                    "employment_stability": 0.35,
                    "wage_fairness": 0.25,
                    "work_life_balance": 0.2,
                    "safety_standards": 0.15,
                    "training_opportunities": 0.05,
                },
                constraints=["minimum_wage", "safety_standards", "working_hours"],
                priority=0.20,
            ),
            "customers": StakeholderGroup(
                name="Customers",
                interests=["service_quality", "price_competitiveness", "reliability", "sustainability"],
                utility_weights={
                    "service_quality": 0.3,
                    "price_competitiveness": 0.25,
                    "delivery_reliability": 0.25,
                    "product_quality": 0.15,
                    "environmental_impact": 0.05,
                },
                constraints=["service_level_agreement", "quality_standards", "fair_pricing"],
                priority=0.25,
            ),
            "environment": StakeholderGroup(
                name="Environment",
                interests=["carbon_reduction", "waste_minimization", "biodiversity", "renewable_energy"],
                utility_weights={
                    "carbon_emissions": 0.35,
                    "waste_generation": 0.25,
                    "energy_efficiency": 0.2,
                    "resource_conservation": 0.15,
                    "ecosystem_protection": 0.05,
                },
                constraints=["emission_limits", "waste_disposal_regulations", "renewable_energy_targets"],
                priority=0.15,
            ),
            "community": StakeholderGroup(
                name="Community",
                interests=["local_employment", "economic_development", "social_impact", "infrastructure"],
                utility_weights={
                    "local_jobs": 0.3,
                    "economic_contribution": 0.25,
                    "social_welfare": 0.2,
                    "infrastructure_development": 0.15,
                    "cultural_preservation": 0.1,
                },
                constraints=["local_hiring_requirements", "community_investment", "noise_pollution_limits"],
                priority=0.15,
            ),
        }

    def _initialize_ethical_principles(self):
        return {
            "sustainability": EthicalPrinciple(
                name="Sustainability",
                description="Minimize environmental impact while maintaining economic viability.",
                constraints=["carbon_neutrality", "waste_reduction", "renewable_energy"],
                metrics=["carbon_footprint", "waste_generated", "energy_consumption"],
                minimum_threshold=0.8,
                penalty_weight=0.3,
            ),
            "equity": EthicalPrinciple(
                name="Equity",
                description="Support fair treatment of stakeholders and communities.",
                constraints=["fair_wages", "equal_opportunity", "community_benefits"],
                metrics=["wage_distribution", "employment_diversity", "community_impact"],
                minimum_threshold=0.7,
                penalty_weight=0.25,
            ),
            "transparency": EthicalPrinciple(
                name="Transparency",
                description="Encourage openness and accountability in decision-making.",
                constraints=["disclosure_requirements", "stakeholder_communication", "audit_trails"],
                metrics=["information_disclosure", "decision_transparency", "reporting_quality"],
                minimum_threshold=0.9,
                penalty_weight=0.2,
            ),
            "responsibility": EthicalPrinciple(
                name="Responsibility",
                description="Account for social and environmental impacts.",
                constraints=["liability_coverage", "remediation_funds", "safety_standards"],
                metrics=["safety_record", "incident_response", "remediation_effectiveness"],
                minimum_threshold=0.85,
                penalty_weight=0.25,
            ),
        }

    def _initialize_ethical_constraints(self):
        return [
            EthicalConstraint(
                name="maximum_carbon_emissions",
                type="hard",
                description="Annual carbon emissions must not exceed the stated limit.",
                evaluation_function="calculate_total_emissions",
                threshold=10000,
                penalty_factor=10.0,
                sense="upper",
            ),
            EthicalConstraint(
                name="minimum_service_level",
                type="hard",
                description="The modeled service-quality indicator must meet its threshold.",
                evaluation_function="calculate_service_level",
                threshold=0.95,
                penalty_factor=8.0,
            ),
            EthicalConstraint(
                name="fair_wage_distribution",
                type="soft",
                description="Wage levels across selected hubs should be relatively equitable.",
                evaluation_function="calculate_wage_equity",
                threshold=0.8,
                penalty_factor=3.0,
            ),
            EthicalConstraint(
                name="local_employment_minimum",
                type="soft",
                description="The average local-employment indicator should meet its threshold.",
                evaluation_function="calculate_local_employment",
                threshold=0.6,
                penalty_factor=4.0,
            ),
            EthicalConstraint(
                name="community_investment_minimum",
                type="preference",
                description="The modeled community-investment share should meet its threshold.",
                evaluation_function="calculate_community_investment",
                threshold=0.02,
                penalty_factor=2.0,
            ),
        ]

    def _initialize_network_data(self):
        self.coordinates = self._generate_global_coordinates()
        self.distance_matrix = self._calculate_global_distance_matrix()
        self.demand_matrix = self._generate_global_demand_matrix()
        self.hub_costs = self._generate_global_hub_costs()
        self.capacity_limits = self._generate_global_capacity_limits()
        self.emission_factors = self._generate_emission_factors()
        self.social_impact_factors = self._generate_social_impact_factors()
        self.wage_levels = self._generate_wage_levels()
        self.community_data = self._generate_community_data()

    def _generate_global_coordinates(self):
        """Return longitude-latitude coordinates for the illustrative locations."""
        coordinates = [
            [-74.0, 40.7], [-118.2, 34.1], [-87.6, 41.9], [-122.3, 47.6],
            [2.3, 48.9], [9.9, 53.6], [-0.1, 51.5], [12.5, 55.7],
            [121.5, 31.2], [139.7, 35.7], [103.8, 1.3], [114.2, 22.3],
            [-46.6, -23.5], [-70.7, -33.5], [18.4, -33.9], [31.2, 30.1],
            [151.2, -33.9], [-123.1, 49.3], [-3.7, 40.4], [127.0, 37.6],
        ]
        if self.num_nodes > len(coordinates):
            raise ValueError("The illustrative coordinate set supports at most 20 nodes.")
        return np.asarray(coordinates[:self.num_nodes], dtype=float)

    def _get_node_region(self, node_id):
        lon, lat = self.coordinates[node_id]
        if -130 <= lon <= -60 and 25 <= lat <= 55:
            return "north_america"
        if -10 <= lon <= 40 and 35 <= lat <= 70:
            return "europe"
        if 60 <= lon <= 150 and -10 <= lat <= 55:
            return "asia"
        if -80 <= lon <= -35 and -55 <= lat <= 10:
            return "south_america"
        if -20 <= lon <= 50 and -35 <= lat <= 40:
            return "africa"
        if 110 <= lon <= 160 and -45 <= lat <= -10:
            return "oceania"
        return "other"

    def _calculate_global_distance_matrix(self):
        """Calculate symmetric great-circle distances in kilometres."""
        distances = np.zeros((self.num_nodes, self.num_nodes), dtype=float)
        for i in range(self.num_nodes):
            for j in range(i + 1, self.num_nodes):
                lon1, lat1 = np.radians(self.coordinates[i])
                lon2, lat2 = np.radians(self.coordinates[j])
                dlat = lat2 - lat1
                dlon = lon2 - lon1
                a = (
                    np.sin(dlat / 2) ** 2
                    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
                )
                a = np.clip(a, 0.0, 1.0)
                distances[i, j] = 6371.0 * 2 * np.arcsin(np.sqrt(a))
                distances[j, i] = distances[i, j]
        return distances

    def _generate_global_demand_matrix(self):
        """Generate directional synthetic demand between distinct locations."""
        multipliers = {
            "north_america": 1.2,
            "europe": 1.1,
            "asia": 1.3,
            "south_america": 0.8,
            "africa": 0.6,
            "oceania": 0.7,
            "other": 1.0,
        }
        demand = np.zeros((self.num_nodes, self.num_nodes), dtype=float)
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    base = random.randint(100, 800)
                    regional = multipliers[self._get_node_region(i)] * multipliers[self._get_node_region(j)]
                    distance_factor = 1.0 / (1.0 + self.distance_matrix[i, j] / 5000.0)
                    demand[i, j] = int(base * regional * distance_factor)
        return demand

    def _generate_global_hub_costs(self):
        costs = np.random.uniform(150000, 500000, self.num_nodes)
        for i in range(self.num_nodes):
            region = self._get_node_region(i)
            if region in ("north_america", "europe"):
                costs[i] *= 1.3
            elif region == "asia":
                costs[i] *= 1.1
            else:
                costs[i] *= 0.7
        for hub in [0, 1, 4, 8, 9]:
            if hub < self.num_nodes:
                costs[hub] *= 1.5
        return costs.astype(int)

    def _generate_global_capacity_limits(self):
        capacities = np.random.uniform(15000, 50000, self.num_nodes)
        for i in range(self.num_nodes):
            region = self._get_node_region(i)
            if region in ("north_america", "europe"):
                capacities[i] *= 1.4
            elif region == "asia":
                capacities[i] *= 1.2
            else:
                capacities[i] *= 0.8
        return capacities.astype(int)

    def _generate_emission_factors(self):
        """Estimate kilograms of CO2 per container on each direct route."""
        factors = np.zeros((self.num_nodes, self.num_nodes), dtype=float)
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    distance = self.distance_matrix[i, j]
                    mode_adjustment = 0.6 if distance > 3000 else 1.4 if distance < 500 else 1.0
                    factors[i, j] = 150 * (distance / 1000) * mode_adjustment
        return factors

    def _generate_social_impact_factors(self):
        factors = {}
        for i in range(self.num_nodes):
            region = self._get_node_region(i)
            employment = 0.8 if region in ("north_america", "europe") else 1.2 if region == "asia" else 1.5
            community = 0.6 if region in ("north_america", "europe") else 1.4
            factors[i] = {
                "employment_impact": employment,
                "community_development": community,
                "skill_development": 1.0,
                "infrastructure_need": 1.0,
            }
        return factors

    def _generate_wage_levels(self):
        ranges = {
            "north_america": (15, 25),
            "europe": (12, 22),
            "asia": (3, 8),
            "south_america": (4, 10),
            "africa": (2, 6),
            "oceania": (5, 15),
            "other": (5, 15),
        }
        return np.array([
            random.uniform(*ranges[self._get_node_region(i)])
            for i in range(self.num_nodes)
        ])

    def _generate_community_data(self):
        data = {}
        for i in range(self.num_nodes):
            region = self._get_node_region(i)
            if region in ("north_america", "europe"):
                need = random.uniform(0.2, 0.4)
            elif region == "asia":
                need = random.uniform(0.4, 0.6)
            else:
                need = random.uniform(0.6, 0.9)
            data[i] = {
                "need_index": need,
                "population_density": random.uniform(100, 10000),
                "education_level": random.uniform(0.3, 0.9),
                "infrastructure_quality": random.uniform(0.3, 0.9),
            }
        return data

    def _generate_ethical_hub_combination(self, num_hubs):
        """Sample hubs from locations with relatively high modeled ethical scores."""
        mean_wage = np.mean(self.wage_levels)
        mean_emissions = np.mean(self.emission_factors)
        hub_scores = []
        for i in range(self.num_nodes):
            average_emissions = np.mean([
                self.emission_factors[i, j]
                for j in range(self.num_nodes)
                if i != j
            ])
            wage_score = 1.0 - abs(self.wage_levels[i] - mean_wage) / mean_wage
            emission_score = 1.0 - average_emissions / mean_emissions if mean_emissions else 0.0
            score = (
                0.3 * self.social_impact_factors[i]["employment_impact"]
                + 0.2 * self.community_data[i]["need_index"]
                + 0.2 * wage_score
                + 0.3 * emission_score
            )
            hub_scores.append((i, score))
        hub_scores.sort(key=lambda item: item[1], reverse=True)
        top_candidates = [node for node, _ in hub_scores[:num_hubs * 2]]
        return random.sample(top_candidates, num_hubs)

    def _calculate_spoke_assignments(self, hub_configuration):
        """Assign each non-hub to its least-cost hub; hubs serve themselves."""
        assignments = {}
        for node in range(self.num_nodes):
            if node in hub_configuration:
                assignments[node] = node
                continue
            costs = []
            for hub in hub_configuration:
                same_region_discount = (
                    0.1
                    if self._get_node_region(node) == self._get_node_region(hub)
                    else 0.0
                )
                costs.append(self.distance_matrix[node, hub] * (1.0 - same_region_discount))
            assignments[node] = hub_configuration[int(np.argmin(costs))]
        return assignments

    def _generate_candidate_solutions(self):
        candidates = []
        for num_hubs in range(3, min(8, self.num_nodes)):
            for _ in range(5):
                hubs = self._generate_ethical_hub_combination(num_hubs)
                assignments = self._calculate_spoke_assignments(hubs)
                candidates.append({
                    "hub_configuration": hubs,
                    "spoke_assignments": assignments,
                })
        if not candidates:
            raise ValueError("Candidate generation requires at least four nodes.")
        return candidates

    def _calculate_network_performance(self, solution):
        hubs = solution["hub_configuration"]
        assignments = solution["spoke_assignments"]
        fixed_cost = sum(self.hub_costs[hub] for hub in hubs)
        total_flow = float(np.sum(self.demand_matrix))
        max_distance = float(np.max(self.distance_matrix))
        variable_cost = 0.0
        service_total = 0.0
        total_emissions_kg = 0.0

        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i == j:
                    continue
                hub_i = assignments[i]
                hub_j = assignments[j]
                flow = self.demand_matrix[i, j]
                route_distance = (
                    self.distance_matrix[i, hub_i]
                    + 0.75 * self.distance_matrix[hub_i, hub_j]
                    + 0.75 * self.distance_matrix[hub_j, j]
                )
                variable_cost += flow * route_distance
                service_total += flow * max(0.0, 1.0 - route_distance / max_distance)
                route_emissions = (
                    self.emission_factors[i, hub_i]
                    + 0.75 * self.emission_factors[hub_i, hub_j]
                    + 0.75 * self.emission_factors[hub_j, j]
                )
                total_emissions_kg += flow * route_emissions

        total_cost = fixed_cost + variable_cost
        service_quality = service_total / total_flow if total_flow else 0.0
        total_emissions_tons = total_emissions_kg / 1000.0
        carbon_performance = float(np.clip(1.0 - total_emissions_tons / 50000.0, 0.0, 1.0))

        total_employment = sum(
            self.social_impact_factors[hub]["employment_impact"]
            * self.capacity_limits[hub] / 10000
            for hub in hubs
        )
        community_impact = sum(
            self.community_data[hub]["need_index"]
            * self.capacity_limits[hub] / 10000
            for hub in hubs
        )
        hub_wages = [self.wage_levels[hub] for hub in hubs]
        wage_mean = float(np.mean(hub_wages))
        wage_equity = float(np.clip(1.0 - np.std(hub_wages) / wage_mean, 0.0, 1.0)) if wage_mean else 0.0

        local_shares = []
        for hub in hubs:
            region = self._get_node_region(hub)
            assigned_nodes = [node for node in range(self.num_nodes) if assignments[node] == hub]
            if assigned_nodes:
                local_count = sum(self._get_node_region(node) == region for node in assigned_nodes)
                local_shares.append(local_count / len(assigned_nodes))
        local_employment = float(np.mean(local_shares)) if local_shares else 0.0
        community_investment = 0.02

        return {
            "total_cost": total_cost,
            "fixed_cost": fixed_cost,
            "variable_cost": variable_cost,
            "service_quality": service_quality,
            "total_emissions_tons": total_emissions_tons,
            "carbon_emissions": carbon_performance,
            "employment_stability": float(np.clip(total_employment / 100.0, 0.0, 1.0)),
            "wage_fairness": wage_equity,
            "local_jobs": local_employment,
            "economic_contribution": float(np.clip(community_impact / 50.0, 0.0, 1.0)),
            "social_welfare": float(np.clip(community_impact / 50.0, 0.0, 1.0)),
            "cost_efficiency": float(np.clip(1.0 - total_cost / 10000000.0, 0.0, 1.0)),
            "revenue_growth": service_quality,
            "market_expansion": len(hubs) / self.num_nodes,
            "risk_management": wage_equity,
            "work_life_balance": 0.8,
            "safety_standards": 0.9,
            "training_opportunities": 0.7,
            "service_quality_indicator": service_quality,
            "price_competitiveness": float(np.clip(1.0 - total_cost / 10000000.0, 0.0, 1.0)),
            "delivery_reliability": service_quality,
            "product_quality": 0.9,
            "environmental_impact": carbon_performance,
            "waste_generation": 0.8,
            "energy_efficiency": 0.8,
            "resource_conservation": 0.7,
            "ecosystem_protection": 0.8,
            "infrastructure_development": float(np.clip(community_impact / 50.0, 0.0, 1.0)),
            "cultural_preservation": 0.8,
            "community_investment": community_investment,
        }

    def _calculate_stakeholder_utilities(self, solution):
        performance = self._calculate_network_performance(solution)
        utilities = {}
        for stakeholder_name, stakeholder in self.stakeholders.items():
            utilities[stakeholder_name] = sum(
                performance.get(metric, 0.0) * weight
                for metric, weight in stakeholder.utility_weights.items()
            )
        return utilities

    def _calculate_total_emissions(self, solution):
        return self._calculate_network_performance(solution)["total_emissions_tons"]

    def _calculate_service_level(self, solution):
        return self._calculate_network_performance(solution)["service_quality"]

    def _calculate_wage_equity(self, solution):
        return self._calculate_network_performance(solution)["wage_fairness"]

    def _calculate_local_employment(self, solution):
        return self._calculate_network_performance(solution)["local_jobs"]

    def _calculate_community_investment(self, solution):
        return self._calculate_network_performance(solution)["community_investment"]

    def _evaluate_constraint(self, solution, constraint):
        evaluators = {
            "calculate_total_emissions": self._calculate_total_emissions,
            "calculate_service_level": self._calculate_service_level,
            "calculate_wage_equity": self._calculate_wage_equity,
            "calculate_local_employment": self._calculate_local_employment,
            "calculate_community_investment": self._calculate_community_investment,
        }
        return evaluators[constraint.evaluation_function](solution)

    def _evaluate_ethical_compliance(self, solution):
        compliance = {}
        total_penalty = 0.0
        type_multiplier = {"hard": 1.0, "soft": 0.5, "preference": 0.25}
        for constraint in self.ethical_constraints:
            value = self._evaluate_constraint(solution, constraint)
            if constraint.sense == "upper":
                violation = max(0.0, value - constraint.threshold) / constraint.threshold
                satisfied = value <= constraint.threshold
            else:
                violation = max(0.0, constraint.threshold - value) / constraint.threshold
                satisfied = value >= constraint.threshold
            penalty = violation * constraint.penalty_factor * type_multiplier[constraint.type]
            compliance[constraint.name] = {
                "value": value,
                "threshold": constraint.threshold,
                "sense": constraint.sense,
                "compliance": satisfied,
                "penalty": penalty,
            }
            total_penalty += penalty
        satisfied_count = sum(item["compliance"] for item in compliance.values())
        compliance["overall_compliance"] = satisfied_count / len(self.ethical_constraints)
        compliance["total_penalty"] = total_penalty
        return compliance

    def _calculate_overall_score(self, utilities, compliance):
        weighted_utility = sum(
            utilities[name] * stakeholder.priority
            for name, stakeholder in self.stakeholders.items()
        )
        return weighted_utility - compliance["total_penalty"]

    def _analyze_trade_offs(self, utilities):
        values = list(utilities.values())
        maximum = max(values)
        minimum = min(values)
        gap = maximum - minimum
        return {
            "utility_variance": float(np.var(values)),
            "utility_gap": gap,
            "equity_score": 1.0 - gap / maximum if maximum > 0 else 0.0,
        }

    def _generate_solution_explanation(self, best_solution):
        hubs = best_solution["solution"]["hub_configuration"]
        utilities = best_solution["utilities"]
        compliance = best_solution["compliance"]
        best_group = max(utilities, key=utilities.get)
        lowest_group = min(utilities, key=utilities.get)
        satisfied = sum(
            item["compliance"]
            for key, item in compliance.items()
            if isinstance(item, dict)
        )
        total = len(self.ethical_constraints)
        return (
            f"Selected {len(hubs)} hubs from the generated candidate set; "
            f"highest stakeholder utility: {best_group} ({utilities[best_group]:.3f}); "
            f"lowest stakeholder utility: {lowest_group} ({utilities[lowest_group]:.3f}); "
            f"{satisfied} of {total} modeled thresholds satisfied."
        )

    def _record_solution(self, solution):
        self.solution_history.append({
            "solution": solution,
            "hub_count": len(solution.hub_configuration),
            "overall_score": solution.overall_score,
            "ethical_compliance": solution.ethical_compliance["overall_compliance"],
        })
        self.compliance_history.append({
            "compliance": solution.ethical_compliance,
            "utilities": solution.stakeholder_utilities,
            "trade_offs": solution.trade_offs,
        })

    def optimize_value_aligned_network(self):
        """Evaluate the generated candidates and return the highest-scoring one."""
        print("=== VALUE-ALIGNED NETWORK EVALUATION ===")
        print("1. Generating candidate solutions...")
        candidates = self._generate_candidate_solutions()
        print(f"   Generated {len(candidates)} candidate solutions.")

        print("\n2. Evaluating stakeholder utilities and constraint penalties...")
        evaluated = []
        for candidate in candidates:
            utilities = self._calculate_stakeholder_utilities(candidate)
            compliance = self._evaluate_ethical_compliance(candidate)
            evaluated.append({
                "solution": candidate,
                "utilities": utilities,
                "compliance": compliance,
                "overall_score": self._calculate_overall_score(utilities, compliance),
                "trade_offs": self._analyze_trade_offs(utilities),
            })

        print("\n3. Selecting the highest-scoring generated candidate...")
        best = max(evaluated, key=lambda item: item["overall_score"])
        print(f"   Candidate score: {best['overall_score']:.3f}")
        print(f"   Thresholds satisfied: {best['compliance']['overall_compliance']:.1%}")

        solution = ValueAlignedSolution(
            hub_configuration=best["solution"]["hub_configuration"],
            spoke_assignments=best["solution"]["spoke_assignments"],
            stakeholder_utilities=best["utilities"],
            ethical_compliance=best["compliance"],
            overall_score=best["overall_score"],
            trade_offs=best["trade_offs"],
            explanation=self._generate_solution_explanation(best),
        )
        self._record_solution(solution)
        return solution


ethical_framework = ValueAlignedEthicalFramework(num_nodes=20)
print("Framework initialized.")
print(f"Network locations: {ethical_framework.num_nodes}")
print(f"Stakeholder groups: {len(ethical_framework.stakeholders)}")
print(f"Named principles: {len(ethical_framework.ethical_principles)}")
print(f"Evaluated constraints: {len(ethical_framework.ethical_constraints)}")
for name, stakeholder in ethical_framework.stakeholders.items():
    print(f"- {name}: priority {stakeholder.priority:.2f}")

In [ ]:
optimal_solution = ethical_framework.optimize_value_aligned_network()

print("\n=== SELECTED CANDIDATE ===")
print(f"Hub locations (1-based labels): {[hub + 1 for hub in optimal_solution.hub_configuration]}")
print(f"Number of hubs: {len(optimal_solution.hub_configuration)}")
print(f"Net score: {optimal_solution.overall_score:.3f}")
print(f"Thresholds satisfied: {optimal_solution.ethical_compliance['overall_compliance']:.1%}")

print("\n=== STAKEHOLDER UTILITIES ===")
for stakeholder, utility in optimal_solution.stakeholder_utilities.items():
    print(f"{stakeholder}: {utility:.3f}")

print("\n=== CONSTRAINT EVALUATION ===")
for name, result in optimal_solution.ethical_compliance.items():
    if isinstance(result, dict):
        relation = "<=" if result["sense"] == "upper" else ">="
        status = "MET" if result["compliance"] else "NOT MET"
        unit = " metric tonnes" if name == "maximum_carbon_emissions" else ""
        print(
            f"{name}: {result['value']:.4f}{unit} {relation} "
            f"{result['threshold']:.4f}{unit} — {status}; "
            f"penalty {result['penalty']:.3f}"
        )

print("\n=== STAKEHOLDER TRADE-OFFS ===")
for name, value in optimal_solution.trade_offs.items():
    print(f"{name}: {value:.3f}")

print("\n=== INTERPRETATION ===")
print(optimal_solution.explanation)

assigned_nodes = set(optimal_solution.spoke_assignments)
assert assigned_nodes == set(range(ethical_framework.num_nodes))
assert all(
    optimal_solution.spoke_assignments[hub] == hub
    for hub in optimal_solution.hub_configuration
)
assert len(optimal_solution.hub_configuration) == len(set(optimal_solution.hub_configuration))
assert np.isclose(
    optimal_solution.overall_score,
    sum(
        optimal_solution.stakeholder_utilities[name] * stakeholder.priority
        for name, stakeholder in ethical_framework.stakeholders.items()
    ) - optimal_solution.ethical_compliance["total_penalty"],
)
print("Validation checks passed.")

In [ ]:
def visualize_ethical_optimization(ethical_framework, optimal_solution):
    """Plot utilities, threshold attainment, network assignments, and indicators."""
    fig = plt.figure(figsize=(16, 12))
    grid = fig.add_gridspec(2, 2)
    ax1 = fig.add_subplot(grid[0, 0])
    ax2 = fig.add_subplot(grid[0, 1])
    ax3 = fig.add_subplot(grid[1, 0])
    ax4 = fig.add_subplot(grid[1, 1], projection="polar")
    fig.suptitle("Value-Aligned Network Evaluation", fontsize=16, fontweight="bold")

    stakeholders = list(optimal_solution.stakeholder_utilities)
    utilities = list(optimal_solution.stakeholder_utilities.values())
    bars = ax1.bar(stakeholders, utilities, color=sns.color_palette("husl", len(stakeholders)))
    ax1.set_title("Stakeholder Utility")
    ax1.set_ylabel("Utility score")
    ax1.tick_params(axis="x", rotation=35)
    ax1.grid(axis="y", alpha=0.3)
    for bar, utility in zip(bars, utilities):
        ax1.annotate(
            f"{utility:.3f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )
    ax1.text(
        0.98,
        0.95,
        f"Utility equity indicator: {optimal_solution.trade_offs['equity_score']:.3f}",
        transform=ax1.transAxes,
        ha="right",
        va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )

    constraint_names = []
    attainment_ratios = []
    for name, result in optimal_solution.ethical_compliance.items():
        if not isinstance(result, dict):
            continue
        threshold = result["threshold"]
        value = result["value"]
        ratio = threshold / value if result["sense"] == "upper" and value > 0 else (
            value / threshold if result["sense"] == "lower" and threshold > 0 else 1.0
        )
        constraint_names.append(name.replace("_", " ").title())
        attainment_ratios.append(ratio)
    colors = ["#4ECDC4" if ratio >= 1 else "#FF6B6B" for ratio in attainment_ratios]
    ax2.barh(constraint_names, attainment_ratios, color=colors, alpha=0.85)
    ax2.axvline(1.0, color="black", linestyle="--", label="Threshold met")
    ax2.set_title("Constraint Attainment Ratio")
    ax2.set_xlabel("Attainment ratio (1.0 meets the threshold)")
    ax2.legend()
    ax2.grid(axis="x", alpha=0.3)

    coordinates = ethical_framework.coordinates
    hub_set = set(optimal_solution.hub_configuration)
    for node, hub in optimal_solution.spoke_assignments.items():
        if node != hub:
            ax3.plot(
                [coordinates[node, 0], coordinates[hub, 0]],
                [coordinates[node, 1], coordinates[hub, 1]],
                linestyle="--",
                color="gray",
                alpha=0.4,
            )
    spokes = [node for node in range(ethical_framework.num_nodes) if node not in hub_set]
    ax3.scatter(
        coordinates[spokes, 0],
        coordinates[spokes, 1],
        s=65,
        color="lightblue",
        edgecolors="black",
        label="Spoke",
        zorder=3,
    )
    ax3.scatter(
        coordinates[list(hub_set), 0],
        coordinates[list(hub_set), 1],
        s=150,
        marker="s",
        color="seagreen",
        edgecolors="black",
        label="Hub",
        zorder=4,
    )
    for node, (longitude, latitude) in enumerate(coordinates):
        ax3.annotate(str(node + 1), (longitude, latitude), xytext=(3, 3), textcoords="offset points", fontsize=7)
    ax3.set_title("Illustrative Hub-and-Spoke Assignments")
    ax3.set_xlabel("Longitude (degrees)")
    ax3.set_ylabel("Latitude (degrees)")
    ax3.legend()
    ax3.grid(alpha=0.3)

    performance = ethical_framework._calculate_network_performance({
        "hub_configuration": optimal_solution.hub_configuration,
        "spoke_assignments": optimal_solution.spoke_assignments,
    })
    metric_labels = [
        "Cost efficiency",
        "Service quality",
        "Carbon indicator",
        "Employment",
        "Local employment",
        "Wage equity",
    ]
    metric_values = [
        performance["cost_efficiency"],
        performance["service_quality"],
        performance["carbon_emissions"],
        performance["employment_stability"],
        performance["local_jobs"],
        performance["wage_fairness"],
    ]
    angles = np.linspace(0, 2 * np.pi, len(metric_labels), endpoint=False)
    closed_angles = np.append(angles, angles[0])
    closed_values = metric_values + metric_values[:1]
    ax4.plot(closed_angles, closed_values, "o-", linewidth=2, color="#FF6B6B", label="Selected candidate")
    ax4.fill(closed_angles, closed_values, alpha=0.2, color="#FF6B6B")
    ax4.set_xticks(angles)
    ax4.set_xticklabels(metric_labels, fontsize=8)
    ax4.set_ylim(0, 1)
    ax4.set_title("Normalized Performance Indicators", pad=20)
    ax4.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
    ax4.grid(alpha=0.4)

    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()
    return fig


fig = visualize_ethical_optimization(ethical_framework, optimal_solution)

In [ ]:
def analyze_ethical_framework_performance(ethical_framework, optimal_solution):
    """Summarize modeled indicators without treating proxies as measured impacts."""
    solution = {
        "hub_configuration": optimal_solution.hub_configuration,
        "spoke_assignments": optimal_solution.spoke_assignments,
    }
    performance = ethical_framework._calculate_network_performance(solution)

    environmental_impact = np.mean([
        performance["carbon_emissions"],
        performance["energy_efficiency"],
        performance["resource_conservation"],
    ])
    social_impact = np.mean([
        performance["employment_stability"],
        performance["local_jobs"],
        performance["wage_fairness"],
        performance["social_welfare"],
    ])
    economic_impact = np.mean([
        performance["cost_efficiency"],
        performance["service_quality"],
        performance["market_expansion"],
    ])
    sustainability_score = np.mean([
        environmental_impact,
        social_impact,
        economic_impact,
    ])

    print("=== MODELED PERFORMANCE SUMMARY ===")
    print(f"Net candidate score: {optimal_solution.overall_score:.3f}")
    print(f"Constraint thresholds met: {optimal_solution.ethical_compliance['overall_compliance']:.1%}")
    print(f"Stakeholder utility equity indicator: {optimal_solution.trade_offs['equity_score']:.3f}")
    print(f"Estimated emissions: {performance['total_emissions_tons']:.2f} metric tonnes")
    print(f"Service-quality indicator: {performance['service_quality']:.3f}")
    print(f"Carbon performance proxy: {performance['carbon_emissions']:.3f}")

    print("\n=== COMPOSITE INDICATORS ===")
    print(f"Environmental indicator: {environmental_impact:.3f}")
    print(f"Social indicator: {social_impact:.3f}")
    print(f"Economic indicator: {economic_impact:.3f}")
    print(f"Mean of the three indicators: {sustainability_score:.3f}")

    unmet = [
        name
        for name, result in optimal_solution.ethical_compliance.items()
        if isinstance(result, dict) and not result["compliance"]
    ]
    print("\n=== REVIEW POINTS ===")
    if unmet:
        print("Thresholds not met: " + ", ".join(unmet))
        print("Review the relevant modeled indicators and policy thresholds before interpreting the candidate.")
    else:
        print("All modeled thresholds are met for this generated candidate.")
    print("These synthetic indicators do not establish real-world impact or regulatory compliance.")

    return {
        "environmental_indicator": float(environmental_impact),
        "social_indicator": float(social_impact),
        "economic_indicator": float(economic_impact),
        "mean_composite_indicator": float(sustainability_score),
        "overall_compliance": optimal_solution.ethical_compliance["overall_compliance"],
        "stakeholder_equity": optimal_solution.trade_offs["equity_score"],
    }


impact_analysis = analyze_ethical_framework_performance(ethical_framework, optimal_solution)

## Interpreting the framework

### Why this tier is distinct

This tier places stakeholder indicators and policy thresholds alongside network cost and service proxies. The distinction is in the evaluation criteria: candidates are not ranked solely by a financial measure. The implementation operationalizes that idea with weighted stakeholder utilities and penalties, while retaining a transparent candidate-generation and candidate-selection sequence.

### Strengths and limitations

**Potential strengths of the modeling structure**

- It makes stakeholder priorities and constraint thresholds visible.
- It reports constraint attainment separately from the overall score.
- It exposes several trade-offs rather than presenting a single cost measure.
- It provides an interpretable record of why a generated candidate received its score.

**Important limitations**

- The weights, thresholds, proxies, and penalty factors are assumptions; changing them can change the ranking.
- The candidate search is stochastic and samples a restricted set of hub configurations. Its selected candidate is not guaranteed to be globally optimal.
- Penalties do not enforce feasibility, including for constraints labelled `hard`.
- The simplified indicators do not measure actual worker welfare, community outcomes, environmental effects, or legal compliance.
- A composite score can conceal disagreement among stakeholders. Individual utilities and constraint results should therefore be examined as well.

### When this modeling perspective may be useful

A stakeholder-oriented evaluation can help structure early strategic discussions when economic, service, environmental, and social considerations all matter. Before using such a model for decisions, practitioners would need validated data, justified and reviewed weights and thresholds, operational definitions for the indicators, and appropriate governance. The outputs here are an educational illustration, not a recommendation for a real network.

### Key interpretation

The notebook demonstrates how values can be represented in a candidate-scoring process and how resulting trade-offs can be inspected. It does not demonstrate a 27% emissions reduction, a 95% service level, a particular level of stakeholder satisfaction, or guaranteed compliance. Such claims require a defined baseline, validated data, and evidence from the modeled or operational system.